# Tabular Model Arena: Hyperparameter & Baseline Tuning
In this notebook, we validate the performance of 9 independent machine learning algorithms on the structured ISCX dataset. This proves why **XGBoost** was selected as the production champion.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

plt.style.use('ggplot')

In [ ]:
# Preprocess Data (Mimicking tabular_pipeline.py)
df = pd.read_csv('../data/iscx_phishing/All.csv', low_memory=False)
y = df['URL_Type_obf_Type'].apply(lambda x: 0 if str(x).strip().lower() == 'benign' else 1)
X = df.drop(columns=['URL_Type_obf_Type'])
X = X.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
X = X.iloc[:, :79]

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_imputed, y, test_size=0.2, random_state=42)

In [ ]:
# Train and compare Tree-based Ensembles
rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42)
xgb = XGBClassifier(eval_metric='logloss', random_state=42)

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]
xgb_probs = xgb.predict_proba(X_test)[:, 1]

rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)

plt.figure(figsize=(8, 6))
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf_probs):.4f})')
plt.plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUC = {roc_auc_score(y_test, xgb_probs):.4f})', color='purple')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) - Tabular ISCX')
plt.legend(loc='lower right')
plt.show()